# 06b — Experiment Tracking

You'll train 100+ models before finding a good one. Without tracking, you'll forget which hyperparameters worked, which data version you used, and why you abandoned an approach three weeks ago. Experiment tracking solves this — it's version control for your ML experiments.

---
## 1 · What to Track

Every experiment should record:

| Category | Examples |
|---|---|
| **Hyperparameters** | learning rate, n_estimators, batch size, dropout |
| **Metrics** | accuracy, F1, loss, AUC, latency |
| **Artifacts** | saved model file, confusion matrix plot, feature importance |
| **Data version** | which dataset split, preprocessing version, row count |
| **Code version** | git commit hash, branch name |
| **Environment** | Python version, library versions, GPU type |

The goal: any experiment should be **reproducible** months later.

---
## 2 · MLflow — The Standard

MLflow is the most widely-used open-source experiment tracking tool. It logs everything to a local directory (or a remote server) and gives you a web UI to compare runs.

```bash
pip install mlflow
```

In [ ]:
import mlflow
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split

X, y = load_wine(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

### Basic logging — one run

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

mlflow.set_experiment("wine-classification")

with mlflow.start_run(run_name="rf-baseline"):
    n_est = 100
    max_depth = 5

    mlflow.log_param("n_estimators", n_est)
    mlflow.log_param("max_depth", max_depth)
    mlflow.log_param("model_type", "RandomForest")

    clf = RandomForestClassifier(n_estimators=n_est, max_depth=max_depth, random_state=42)
    clf.fit(X_train, y_train)
    preds = clf.predict(X_test)

    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds, average="weighted")

    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("f1_weighted", f1)

    mlflow.sklearn.log_model(clf, "model")

    print(f"Accuracy: {acc:.3f} | F1: {f1:.3f}")

### Logging artifacts — save plots and files alongside the run

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

with mlflow.start_run(run_name="rf-with-artifacts"):
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("max_depth", None)

    clf = RandomForestClassifier(n_estimators=200, random_state=42)
    clf.fit(X_train, y_train)
    preds = clf.predict(X_test)

    mlflow.log_metric("accuracy", accuracy_score(y_test, preds))
    mlflow.log_metric("f1_weighted", f1_score(y_test, preds, average="weighted"))

    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay.from_predictions(y_test, preds, ax=ax)
    fig.savefig("confusion_matrix.png", dpi=100, bbox_inches="tight")
    plt.close()
    mlflow.log_artifact("confusion_matrix.png")

    mlflow.sklearn.log_model(clf, "model")
    print("Run logged with confusion matrix artifact")

### Running multiple experiments — hyperparameter sweep

This is where tracking really pays off. Train a bunch of models, log everything, then compare in the UI.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

configs = [
    {"name": "rf-50",   "model": RandomForestClassifier(n_estimators=50, random_state=42)},
    {"name": "rf-200",  "model": RandomForestClassifier(n_estimators=200, random_state=42)},
    {"name": "rf-500",  "model": RandomForestClassifier(n_estimators=500, random_state=42)},
    {"name": "lr-l2",   "model": make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=1000))},
    {"name": "lr-l1",   "model": make_pipeline(StandardScaler(), LogisticRegression(C=0.1, penalty="l1", solver="saga", max_iter=1000))},
    {"name": "svm-rbf", "model": make_pipeline(StandardScaler(), SVC(kernel="rbf", probability=True))},
]

for cfg in configs:
    with mlflow.start_run(run_name=cfg["name"]):
        mlflow.log_param("model_name", cfg["name"])

        pipe = cfg["model"]
        pipe.fit(X_train, y_train)
        preds = pipe.predict(X_test)

        acc = accuracy_score(y_test, preds)
        f1 = f1_score(y_test, preds, average="weighted")

        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("f1_weighted", f1)
        mlflow.sklearn.log_model(pipe, "model")

        print(f"{cfg['name']:>8s}  acc={acc:.3f}  f1={f1:.3f}")

### The MLflow UI

Launch it from the terminal:
```bash
mlflow ui --port 5000
```

Then open `http://127.0.0.1:5000`. The dashboard shows:

- **Experiment list**: all your experiments (like folders)
- **Run table**: every run within an experiment, sortable by any metric
- **Run detail page**: parameters, metrics over time, artifacts (model files, plots)
- **Compare view**: select multiple runs → side-by-side metric comparison, parallel coordinate plots

You can filter, sort by accuracy, find the best run, and download its model — all from the browser.

---
## 3 · Comparing Experiments Programmatically

In [ ]:
import pandas as pd

experiment = mlflow.get_experiment_by_name("wine-classification")
runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

results = runs[["run_id", "params.model_name", "metrics.accuracy", "metrics.f1_weighted"]].copy()
results.columns = ["run_id", "model", "accuracy", "f1"]
results = results.dropna(subset=["model"]).sort_values("accuracy", ascending=False)
print(results.to_string(index=False))

best_run_id = results.iloc[0]["run_id"]
print(f"\nBest run: {best_run_id}")

### Load the best model back

In [ ]:
best_model = mlflow.sklearn.load_model(f"runs:/{best_run_id}/model")
preds = best_model.predict(X_test)
print(f"Loaded best model — accuracy: {accuracy_score(y_test, preds):.3f}")

---
## 4 · Model Registry

Once you find the best model, you want to **register** it — give it a name and version, and move it through stages:

```
None → Staging → Production → Archived
```

This way your serving infrastructure always pulls from "the production model" without hardcoding run IDs.

In [ ]:
from mlflow import MlflowClient

model_uri = f"runs:/{best_run_id}/model"
registered = mlflow.register_model(model_uri, "wine-classifier")
print(f"Registered: {registered.name} v{registered.version}")

client = MlflowClient()
client.set_registered_model_alias("wine-classifier", "champion", registered.version)

champion = mlflow.sklearn.load_model("models:/wine-classifier@champion")
print(f"Champion model accuracy: {accuracy_score(y_test, champion.predict(X_test)):.3f}")

---
## 5 · Weights & Biases (W&B) — Overview

W&B is a cloud-hosted alternative to MLflow. The API is similarly simple, but everything is stored on their servers with a polished web dashboard, real-time collaboration, and built-in hyperparameter sweep tools.

```bash
pip install wandb
wandb login
```

In [ ]:
import wandb

wandb.init(
    project="wine-classification",
    config={"n_estimators": 200, "model": "RandomForest"},
)

clf = RandomForestClassifier(n_estimators=200, random_state=42)
clf.fit(X_train, y_train)
preds = clf.predict(X_test)

wandb.log({
    "accuracy": accuracy_score(y_test, preds),
    "f1_weighted": f1_score(y_test, preds, average="weighted"),
})

wandb.finish()

### MLflow vs W&B

| Feature | MLflow | W&B |
|---|---|---|
| Hosting | Self-hosted (or Databricks) | Cloud (free tier available) |
| Setup | `mlflow ui` locally | `wandb login` + cloud dashboard |
| Collaboration | Manual (share server URL) | Built-in (team workspaces) |
| Sweeps | Not built-in | `wandb.sweep()` for hyperparameter search |
| Model Registry | ✓ | ✓ (Model Registry + Artifacts) |
| Cost | Free (self-hosted) | Free tier, then paid for teams |

**Pick MLflow** if you want full control and self-hosting. **Pick W&B** if you want a polished cloud experience with zero infra setup.

---
## Summary

| Concept | Key Takeaway |
|---|---|
| Why track | You'll forget what worked without systematic logging |
| What to track | Params, metrics, artifacts, data version, code version |
| MLflow | `log_param`, `log_metric`, `log_model` inside `start_run()` |
| Compare runs | `search_runs()` → sort by metric → find best |
| Model Registry | Name + version + stage = clean model management |
| W&B | Cloud-hosted alternative with sweeps and collaboration |